# 03 — Qwen3.8-27B (free) via OpenRouter — retrieval + summarization (accuracy-only)

Hosted-API arm. Verified 2026-10-03 against `GET /api/v1/models`: **both** `qwen/qwen3.8-27b` (paid, 1M ctx) and `qwen/qwen3.8-27b:free` (price $0/$0, 262K ctx, `text+image+video->text`) exist. This notebook uses the `:free` id.

**Rules for this arm:** accuracy/quality metrics only (hit rate, F1, ROUGE). NEVER compare its latency/VRAM against local runs (AGENTS.md §15). Free tier ≈ 200 req/day — this notebook makes ~18 calls by default (1 verify + 9 retrieval + 8 summarization). Keep `temperature=0`, SAME question string both arms (`+ " Output the answer directly."` for retrieval), only the context representation changes.

Key comes from Kaggle Secret `OPENROUTER_API_KEY` (Add-ons → Secrets). It is never hardcoded. You said you will provide it in the Kaggle environment — set the secret, then Run All.

In [ ]:
# Cell 1 — config
MODEL_ID = "qwen/qwen3.8-27b:free"  # verified free: $0 in/out, 262K ctx, vision in
MODEL_HF_TOKENIZER = "Qwen/Qwen3.8-27B"  # tokenizer/processor-only download (CPU, tiny, no weights)
BUDGETS = [{"tag": "low", "canvas_w": 600}, {"tag": "high", "canvas_w": 1100}]
CONTEXTS = [{"tag": "short", "words": 350}, {"tag": "med", "words": 1500}, {"tag": "long", "words": 3000}]
N_SUMM = 4          # CNN/DailyMail articles (each = 1 text + 1 image call). Lower if quota errors.
SUMM_WORDS = 800    # truncate articles to this many words so the rendered image stays legible
GEN_RET = {"temperature": 0.0, "max_new_tokens": 256}
GEN_SUMM = {"temperature": 0.0, "max_new_tokens": 256}
OUT = "/kaggle/working/results/raw/qwen38_openrouter.jsonl"
print(MODEL_ID, BUDGETS, [c['tag'] for c in CONTEXTS], f"N_SUMM={N_SUMM}")

In [ ]:
# Cell 2 — deps + key + reachability check (1 cheap text-only call)
!pip install -q openai pillow pandas rouge-score datasets huggingface_hub transformers 2>&1 | tail -1
import os, time, json
from openai import OpenAI

key = os.environ.get("OPENROUTER_API_KEY", "")
try:
    from kaggle_secrets import UserSecretsClient
    key = key or UserSecretsClient().get_secret("OPENROUTER_API_KEY")
except Exception as e:
    print("kaggle_secrets unavailable:", e)
assert key, "Set Kaggle Secret OPENROUTER_API_KEY first (Add-ons → Secrets)."

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)
r = client.chat.completions.create(model=MODEL_ID,
    messages=[{"role": "user", "content": "Reply with exactly: OK"}],
    temperature=0.0, max_tokens=8, timeout=120)
print("reachability:", r.choices[0].message.content.strip(), "|", r.model, "| usage:", r.usage)

In [ ]:
# Cell 3 — renderer + samples + metrics + API helpers (PIL standalone; same logic as 01/02)
import os as _os
_os.environ["HF_HOME"] = "/tmp/hf_home"  # tokenizer/processor downloads to /tmp, not working dir
import base64, io, random, re, string, textwrap, unicodedata
from PIL import Image, ImageDraw, ImageFont

def normalize(t):
    t = unicodedata.normalize("NFKC", t or "")
    return re.sub(r"[ \t]+", " ", t.replace("\r\n", "\n")).strip()

def render(text, width=800, font_size=18, ls=6, wrap=80, pad=30):
    text = normalize(text)
    try: font = ImageFont.truetype("arial.ttf", font_size)
    except IOError: font = ImageFont.load_default()
    lines = []
    for p in text.split("\n\n"):
        lines.extend(textwrap.wrap(p, width=wrap)); lines.append("")
    img = Image.new("RGB", (width, 2*pad + max(len(lines)*(font_size+ls), 100)), "white")
    d = ImageDraw.Draw(img); y = pad
    for ln in lines:
        if ln: d.text((pad, y), ln, fill="black", font=font)
        y += font_size + ls
    return img

def to_data_uri(img, max_h=2600):
    """Downscale tall renders + JPEG-encode to keep the API payload sane. Logged as image_sent_size."""
    if img.height > max_h:
        img = img.resize((int(img.width*max_h/img.height), max_h))
    buf = io.BytesIO(); img.convert("RGB").save(buf, "JPEG", quality=72)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode(), img.size

DIST = ("The thermal radiator on the secondary boom radiates excess heat into cold space. "
        "Star tracking sensor arrays refresh coordinate quaternions at thirty hertz. ")
def make_sample(words, seed):
    rng = random.Random(seed)
    code = f"CIPHER-{rng.randint(1000, 9999)}-OMEGA"
    needle = f"Special authorization key {code} was recorded in emergency protocol ledger section 9."
    paras, n = [], 0
    while n < words:
        p = " ".join(rng.choice([DIST]) for _ in range(6)); paras.append(p); n += len(p.split())
    w = paras[len(paras)//2].split(); w.insert(len(w)//2, needle)
    paras[len(paras)//2] = " ".join(w)
    return "\n\n".join(paras), code

def norm(s):
    return re.sub(r"\s+", " ", (s or "").lower().translate(str.maketrans("", "", string.punctuation))).strip()
def tok_f1(pred, gt):
    p, g = set(norm(pred).split()), set(norm(gt).split())
    if not p or not g: return 0.0
    tp = len(p & g); prec, rec = tp/len(p), tp/len(g)
    return 0.0 if prec+rec == 0 else 2*prec*rec/(prec+rec)

def chat(messages, max_tokens, retries=4):
    """One chat call with backoff for free-tier 429/502. Returns (text, usage_dict)."""
    import openai
    for a in range(retries):
        try:
            r = client.chat.completions.create(model=MODEL_ID, messages=messages,
                temperature=0.0, max_tokens=max_tokens, timeout=300)
            u = r.usage
            usage = {"prompt": getattr(u, 'prompt_tokens', None), "completion": getattr(u, 'completion_tokens', None)} if u else {}
            return (r.choices[0].message.content or "", usage)
        except openai.RateLimitError:
            time.sleep(20*(a+1)); continue
        except openai.APIStatusError as e:
            if getattr(e, 'status_code', 0) in (502, 503) and a < retries-1:
                time.sleep(15*(a+1)); continue
            raise
    raise RuntimeError("API retries exhausted (free-tier quota? lower N_SUMM / retry tomorrow).")

# Optional local accounting: HF tokenizer/processor download is CPU-only (no weights, tiny).
# Gives real m/q (+ decoder-visible image length when the processor supports images).
# On failure (gated/no net) we keep API usage tokens and log CR as null — never estimate.
tok, proc = None, None
try:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(MODEL_HF_TOKENIZER, trust_remote_code=True)
    print("tokenizer OK:", MODEL_HF_TOKENIZER)
except Exception as e:
    print("no local tokenizer (m/q will be null, API usage still logged):", str(e)[:160])
try:
    from transformers import AutoProcessor
    proc = AutoProcessor.from_pretrained(MODEL_HF_TOKENIZER, trust_remote_code=True)
    print("processor OK")
except Exception as e:
    print("no local processor (image decoder length via API usage only):", str(e)[:160])

In [ ]:
# Cell 4 — retrieval A/B loop: text arm vs image arm (9 API calls). Accuracy-only.
import os
from datetime import datetime, timezone
os.makedirs(os.path.dirname(OUT), exist_ok=True)
QSUFFIX = " Output the answer directly."
n = 0
for ci, c in enumerate(CONTEXTS):
    context, gt = make_sample(c["words"], seed=100+ci)
    question = "What special authorization key was recorded in emergency protocol ledger section 9?" + QSUFFIX
    m = len(tok(context, add_special_tokens=False).input_ids) if tok else None
    q = len(tok(question, add_special_tokens=False).input_ids) if tok else None
    # Arm A: text
    pred_t, use_t = chat([{"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}], GEN_RET["max_new_tokens"])
    for b in BUDGETS:
        img = render(context, width=b["canvas_w"])
        uri, sent_size = to_data_uri(img)
        dec_img_len = None
        if proc is not None:
            try:
                pin = proc(text=f"Question: {question}", images=[img], return_tensors="pt")
                dec_img_len = int(pin["input_ids"].shape[-1])
            except Exception as e:
                print("processor image encode failed:", str(e)[:120])
        pred_i, use_i = chat([{"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": uri}},
            {"type": "text", "text": f"Question: {question}"}]}], GEN_RET["max_new_tokens"])
        k = max(dec_img_len - q, 1) if (dec_img_len and q) else None
        rec = {"experiment_id": "qwen38_api_ret", "timestamp": datetime.now(timezone.utc).isoformat(),
               "model": MODEL_ID, "task": "ruler-sniah", "input_mode": "paired-api-accuracy-only",
               "context_tag": c["tag"], "visual_budget_tag": b["tag"], "canvas_w": b["canvas_w"],
               "image_size": f"{img.width}x{img.height}", "image_sent_size": f"{sent_size[0]}x{sent_size[1]}",
               "renderer": "pil-fallback-jpeg72", "context_text_tokens_m": m, "question_tokens_q": q,
               "decoder_image_len_processor": dec_img_len, "visual_tokens_proxy_k": k,
               "CR_context_m_over_k": round(m/k, 3) if (m and k) else None,
               "api_prompt_text": use_t.get("prompt"), "api_prompt_image": use_i.get("prompt"),
               "hit_text": int(norm(gt) in norm(pred_t)), "hit_image": int(norm(gt) in norm(pred_i)),
               "f1_text": round(tok_f1(pred_t, gt), 3), "f1_image": round(tok_f1(pred_i, gt), 3),
               "lat_text_s": None, "lat_image_s": None, "vram_text_gb": None, "vram_image_gb": None,
               "ground_truth": gt, "pred_text": pred_t.strip()[:300], "pred_image": pred_i.strip()[:300]}
        open(OUT, "a").write(json.dumps(rec) + "\n"); n += 1
        print(f"[{n}/6] {c['tag']} x {b['tag']}: m={m} k={k} CR={rec['CR_context_m_over_k']} hitT/I={rec['hit_text']}/{rec['hit_image']}")
print(f"\nRETRIEVAL LOGGED {n} paired rows -> {OUT}")

In [ ]:
# Cell 5 — summarization A/B loop (2*N_SUMM API calls). ROUGE-1/2/L vs reference summary.
from datetime import datetime, timezone
try:
    from datasets import load_dataset
    ds = load_dataset("cnn_dailymail", "3.0.0", split=f"test[:{N_SUMM}]")
    arts = [{"article": r["article"], "ref": r["highlights"]} for r in ds]
    print(f"CNN/DM loaded: {len(arts)} articles")
except Exception as e:
    print("CNN/DM load failed (offline?):", str(e)[:200], "— summarization skipped, retrieval rows already logged.")
    arts = []
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
QSUMM = "Summarize the following document in 3-4 sentences. Output the summary directly."
n2 = 0
for si, a in enumerate(arts):
    doc = " ".join(a["article"].split()[:SUMM_WORDS])
    m = len(tok(doc, add_special_tokens=False).input_ids) if tok else None
    pred_t, _ = chat([{"role": "user", "content": f"Document:\n{doc}\n\n{QSUMM}"}], GEN_SUMM["max_new_tokens"])
    img = render(doc, width=1100)
    uri, sent_size = to_data_uri(img)
    pred_i, _ = chat([{"role": "user", "content": [
        {"type": "image_url", "image_url": {"url": uri}},
        {"type": "text", "text": QSUMM}]}], GEN_SUMM["max_new_tokens"])
    st = scorer.score(a["ref"], pred_t); si_ = scorer.score(a["ref"], pred_i)
    rec = {"experiment_id": "qwen38_api_summ", "timestamp": datetime.now(timezone.utc).isoformat(),
           "model": MODEL_ID, "task": "cnn-dm-summ", "input_mode": "paired-api-accuracy-only",
           "sample_id": si, "context_text_tokens_m": m, "image_size": f"{img.width}x{img.height}",
           "renderer": "pil-fallback-jpeg72",
           "rouge1_text": round(st["rouge1"].fmeasure, 4), "rouge2_text": round(st["rouge2"].fmeasure, 4),
           "rougeL_text": round(st["rougeL"].fmeasure, 4),
           "rouge1_image": round(si_["rouge1"].fmeasure, 4), "rouge2_image": round(si_["rouge2"].fmeasure, 4),
           "rougeL_image": round(si_["rougeL"].fmeasure, 4),
           "lat_text_s": None, "lat_image_s": None, "pred_text": pred_t.strip()[:500], "pred_image": pred_i.strip()[:500]}
    open(OUT, "a").write(json.dumps(rec) + "\n"); n2 += 1
    print(f"[summ {n2}/{len(arts)}] R-L text={rec['rougeL_text']} image={rec['rougeL_image']}")
print(f"SUMMARIZATION LOGGED {n2} rows -> {OUT}")

In [ ]:
# Cell 6 — aggregate from the JSONL (never hand-edit)
import pandas as pd
df = pd.read_json(OUT, lines=True)
ret = df[df.task == "ruler-sniah"]
if len(ret):
    g = ret.groupby("visual_budget_tag").agg(n=("hit_image", "size"), acc_text=("hit_text", "mean"),
        acc_image=("hit_image", "mean"), f1_image=("f1_image", "mean"), cr_ctx=("CR_context_m_over_k", "mean"))
    print("RETRIEVAL (paired, accuracy-only):"); print(g.round(3).to_string())
summ = df[df.task == "cnn-dm-summ"]
if len(summ):
    print("\nSUMMARIZATION (mean ROUGE F):")
    print(summ[["rouge1_text", "rouge2_text", "rougeL_text", "rouge1_image", "rouge2_image", "rougeL_image"]].mean().round(4).to_string())
print("\nGATE: pass iff 6 retrieval rows logged with hit/f1 non-null. CR null is acceptable here (no local processor) — report API usage tokens instead.")